Imports

In [14]:
%pip install -q matplotlib seaborn xgboost scikit-learn pandas numpy


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [28]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import os

Create dataframe and initialize X and Y

In [29]:
path = "data/modeling_dataset.csv"

base_dir = os.path.dirname(os.path.abspath(os.getcwd()))

df = pd.read_csv(os.path.join(base_dir, path))

X = df.drop(columns=["fantasy_points"])

# bucket points into 4 ordered tiers: bench / ok / good / haul                                                                        
y = pd.cut(df["fantasy_points"].clip(lower=0),                                                                                        
            bins=[-1, 1, 4, 8, 100],                   
            labels=[0, 1, 2, 3]).astype(int)   

Print first 5 rows to see some data

In [30]:
df.head(5)

,player,team,opponent,home_away,fantasy_position,season,matchday,date,fantasy_points_prev,fantasy_points_last3_avg,...,pens_made_last3_avg,pens_made_last5_avg,pens_att_prev,pens_att_last3_avg,pens_att_last5_avg,played_prev,played_60_prev,minutes_last3_sum,regular_recently,fantasy_points
0,Aaron Anselmino,Dortmund de,es Villarreal,home,DEF,2025-2026,10,2025-11-25,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1,0,6.0,0,9
1,Aaron Anselmino,Dortmund de,no Bodø/Glimt,home,DEF,2025-2026,13,2025-12-10,9.0,4.5,...,0.0,0.0,0.0,0.0,0.0,1,1,96.0,0,1
2,Aaron Bouwman,nl Ajax,Villarreal es,away,DEF,2025-2026,14,2026-01-20,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1,1,90.0,0,1
3,Aaron Mooy,Celtic sct,de RB Leipzig,home,MID,2022-2023,7,2022-10-11,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1,0,23.0,0,1
4,Aaron Mooy,Celtic sct,ua Shakhtar Donetsk,home,MID,2022-2023,9,2022-10-25,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1,0,48.0,0,1


Display X and y

In [31]:
X.head(5)

,player,team,opponent,home_away,fantasy_position,season,matchday,date,fantasy_points_prev,fantasy_points_last3_avg,...,pens_made_prev,pens_made_last3_avg,pens_made_last5_avg,pens_att_prev,pens_att_last3_avg,pens_att_last5_avg,played_prev,played_60_prev,minutes_last3_sum,regular_recently
0,Aaron Anselmino,Dortmund de,es Villarreal,home,DEF,2025-2026,10,2025-11-25,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1,0,6.0,0
1,Aaron Anselmino,Dortmund de,no Bodø/Glimt,home,DEF,2025-2026,13,2025-12-10,9.0,4.5,...,0.0,0.0,0.0,0.0,0.0,0.0,1,1,96.0,0
2,Aaron Bouwman,nl Ajax,Villarreal es,away,DEF,2025-2026,14,2026-01-20,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1,1,90.0,0
3,Aaron Mooy,Celtic sct,de RB Leipzig,home,MID,2022-2023,7,2022-10-11,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1,0,23.0,0
4,Aaron Mooy,Celtic sct,ua Shakhtar Donetsk,home,MID,2022-2023,9,2022-10-25,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1,0,48.0,0


In [32]:
y.head(5)

0    3
1    0
2    0
3    0
4    0
Name: fantasy_points, dtype: int64

From the X matrix, we can see that there is a lot of significant and insignificant data that are in a nonumeric format.

In [33]:
# Preprocessing for converting text columns into numeric 
# drop name, date, season, team, opponent (text columns)
X_clean = X.drop(columns=["player", "date", "season", "team", "opponent"])

# turning home and away into binary values
X_clean["home_away"] = (X_clean["home_away"] == "home").astype(int)

# every position column, marked 1 if it matches that row and 0 otherwise  
X_clean = pd.get_dummies(X_clean, columns=["fantasy_position"], drop_first=True)                                                      
X_clean = X_clean.astype(float) 

print("Shape: ", X_clean.shape)
print("non-numeric columns left", X_clean.select_dtypes(exclude="number").columns.tolist())
X_clean.head()

Shape:  (16549, 87)
non-numeric columns left []


,home_away,matchday,fantasy_points_prev,fantasy_points_last3_avg,fantasy_points_last5_avg,minutes_prev,minutes_last3_avg,minutes_last5_avg,goals_prev,goals_last3_avg,...,pens_att_prev,pens_att_last3_avg,pens_att_last5_avg,played_prev,played_60_prev,minutes_last3_sum,regular_recently,fantasy_position_FWD,fantasy_position_GK,fantasy_position_MID
0,1.0,10.0,0.0,0.0,0.0,6.0,6.0,6.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,6.0,0.0,0.0,0.0,0.0
1,1.0,13.0,9.0,4.5,4.5,90.0,48.0,48.0,0.0,0.0,...,0.0,0.0,0.0,1.0,1.0,96.0,0.0,0.0,0.0,0.0
2,0.0,14.0,1.0,1.0,1.0,90.0,90.0,90.0,0.0,0.0,...,0.0,0.0,0.0,1.0,1.0,90.0,0.0,0.0,0.0,0.0
3,1.0,7.0,1.0,1.0,1.0,23.0,23.0,23.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,23.0,0.0,0.0,0.0,1.0
4,1.0,9.0,1.0,1.0,1.0,25.0,24.0,24.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,48.0,0.0,0.0,0.0,1.0


In [34]:
# train test split
order = df.sort_values(by="date").index
X_sorted = X_clean.loc[order].reset_index(drop=True)
y_sorted = y.loc[order].reset_index(drop=True)

# train on first 80% of games and last 20% for testing
split_indx = int(len(X_sorted) * 0.8)
X_train, X_test = X_sorted.iloc[:split_indx], X_sorted.iloc[split_indx:]
y_train, y_test = y_sorted.iloc[:split_indx], y_sorted.iloc[split_indx:]

print("Training set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)
 

Training set shape: (13239, 87)
Testing set shape: (3310, 87)


In [35]:
model = LinearRegression()

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

In [36]:
print("MAE:", mean_absolute_error(y_test, y_pred))

print("MSE:", mean_squared_error(y_test, y_pred))

print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))

print("R2:", r2_score(y_test, y_pred))

MAE: 0.659474937271252
MSE: 0.6491882501338766
RMSE: 0.8057221916602996
R2: 0.15603432223659464


In [37]:
# XGBoost model
model = XGBClassifier(
    objective="multi:softprob",
    num_class=4,
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    random_state=42,
)
model.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes f

In [38]:
# show accuracy

pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))
print("Baseline (always predict bucket 0):", (y_test == 0).mean())
print()
print("Confusion matrix (rows = actual, cols = predicted):")
print(confusion_matrix(y_test, pred))

Accuracy: 0.5504531722054381
Baseline (always predict bucket 0): 0.4561933534743202

Confusion matrix (rows = actual, cols = predicted):
[[1124  344   41    1]
 [ 361  623   69    5]
 [ 299  227   74    4]
 [  36   72   29    1]]


In [39]:
# see example predictions 

order = df.sort_values(by="date").index
df_sorted = df.loc[order].reset_index(drop=True)
split_indx = int(len(df_sorted) * 0.8)
df_test = df_sorted.iloc[split_indx:].reset_index(drop=True)

bucket_names = {0: "bench", 1: "ok", 2: "good", 3: "haul"}

results = pd.DataFrame({
    "player":   df_test["player"],
    "opponent": df_test["opponent"],
    "date":     df_test["date"],
    "actual_pts":    df_test["fantasy_points"],
    "actual_bucket": [bucket_names[b] for b in y_test],
    "predicted":     [bucket_names[b] for b in pred],
})

results.sample(20, random_state=1)

,player,opponent,date,actual_pts,actual_bucket,predicted
2453,Mamadou Coulibaly,fr Paris Saint-Germain,2026-02-17,1,bench,ok
1673,Nick Pope,nl PSV,2026-01-21,7,good,ok
630,Raphael Onyedika,Sporting CP pt,2025-11-26,1,bench,ok
792,Bilal Nadir,Union SG be,2025-12-09,1,bench,ok
2501,Piotr Zieliński,Bodø/Glimt no,2026-02-18,1,bench,ok
2721,Leandro Barreiro Martins,Real Madrid es,2026-02-25,2,ok,ok
1702,Mateusz Kochalski,de Eintracht Frankfurt,2026-01-21,4,ok,ok
846,Lucas Bergvall,cz Slavia Prague,2025-12-09,1,bench,ok
93,David Goldar,es Villarreal,2025-11-05,6,good,bench
927,Kevin Mac Allister,fr Marseille,2025-12-09,1,bench,bench


In [40]:
# examples where the model predicted a haul
results[results["predicted"] == "haul"].head(20)

,player,opponent,date,actual_pts,actual_bucket,predicted
119,Erling Haaland,de Dortmund,2025-11-05,6,good,haul
506,Kostas Tzolakis,es Real Madrid,2025-11-26,3,ok,haul
956,Jindřich Staněk,Tottenham Hotspur eng,2025-12-09,9,haul,haul
1009,Anatolii Trubin,it Napoli,2025-12-10,8,good,haul
1108,Erling Haaland,Real Madrid es,2025-12-10,6,good,haul
1454,Philipp Köhn,Real Madrid es,2026-01-20,0,bench,haul
1630,Kauã Santos,Qarabağ az,2026-01-21,4,ok,haul
1824,Jay Gorter,cz Slavia Prague,2026-01-28,3,ok,haul
2111,Marco Sportiello,Union SG be,2026-01-28,4,ok,haul
2312,Julián Álvarez,no Bodø/Glimt,2026-01-28,2,ok,haul
